# CLASSIFICATION

In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv(r'C:\Users\moham\Desktop\Data Science\BeCode\becode_projects\immo-eliza\data\processed\listings_with_distance_clean.csv')

### DATA FORMATTING


In [ ]:
df.columns
df 

Index(['locality', 'region', 'zip_code', 'property_type', 'subtype',
       'price_eur', 'type_of_sale', 'num_rooms', 'living_area_m2',
       'fully_equipped_kitchen', 'furnished', 'terrace', 'terrace_area_m2',
       'garden', 'garden_area_m2', 'land_surface_m2', 'num_facades',
       'swimming_pool', 'state_of_building', 'num_bathrooms', 'dist_train_km',
       'dist_bus_km'],
      dtype='str')

KEEPING ZIP_CODE BECAUSE FOR EXAMPLE 1000 ZIP_CODE IS 90% APARTMENT
(must retest model without zip_code)

In [6]:
#splitting zip code
#use first two digits to get specific info about municipalities/provinces

df['zip_code'] = df['zip_code'] // 100 

X and y variables

In [7]:
#creating target variables by creating new is_house column
#we take every row in property_type that are house which are True
#we convert them to int becoming 1s for House rows and 0s for apartment rows
df['is_house'] = (df['property_type'] == 'House').astype(int)
y = df['is_house']

X = df.drop(columns=['is_house', 'property_type', 'price_eur', 'subtype', 'type_of_sale', 'dist_train_km', 'dist_bus_km', 'region', 'locality', 'state_of_building', 'land_surface_m2'])

# 3. Check the balance (If big imbalance like 90 house vs 10 apartment then model accuracy could be false)
print("Class Distribution:")
print(y.value_counts(normalize=True))

Class Distribution:
is_house
1    0.599896
0    0.400104
Name: proportion, dtype: float64


### DATA SPLITTING


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [9]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 23169 entries, 0 to 23168
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   zip_code                23169 non-null  int64  
 1   num_rooms               23169 non-null  int64  
 2   living_area_m2          23169 non-null  float64
 3   fully_equipped_kitchen  23169 non-null  int64  
 4   furnished               23169 non-null  int64  
 5   terrace                 23169 non-null  int64  
 6   terrace_area_m2         15626 non-null  float64
 7   garden                  23169 non-null  int64  
 8   garden_area_m2          15292 non-null  float64
 9   num_facades             17971 non-null  float64
 10  swimming_pool           23169 non-null  int64  
 11  num_bathrooms           17072 non-null  float64
dtypes: float64(5), int64(7)
memory usage: 2.1 MB


### PREPROCESSING PIPELINE


In [10]:
numeric_features = ['num_rooms', 
                    'living_area_m2', 
                    'terrace_area_m2', 
                    'garden_area_m2', 
                    'num_facades', 
                    'num_bathrooms']

nominal_features = [
    'zip_code'
]

binary_features = [
    'fully_equipped_kitchen',
    'furnished',
    'terrace',  
    'swimming_pool',
    'garden'
]

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

In [12]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

nominal_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

#binary features dont change, they will pass through Columntransformer untouched

In [13]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('nom', nominal_pipeline, nominal_features),
        ('bin', 'passthrough', binary_features) #binary features pass through untouched
    ],
    remainder='drop'
)


# Logistic Regression


In [ ]:
from sklearn.linear_model import LogisticRegression

logreg_model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])



In [15]:
logreg_model_pipeline.fit(X_train, y_train)
logreg_model_pipeline.score(X_train, y_train)

0.8923657944429457

In [16]:
logreg_model_pipeline.score(X_test, y_test)

0.8929650410012948

Scores are too high
Model found a cheat_code, probably because land_surface being 0 for every apartment

### PERMUTATION IMPORTANCE
To check which feature influenced the most

In [ ]:
from sklearn.inspection import permutation_importance

KNN_perm_result = permutation_importance(logreg_model_pipeline, 
                                         X_test, 
                                         y_test, 
                                         scoring='accuracy', 
                                         n_repeats=10, 
                                         random_state=42)

RFC_perm_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance': KNN_perm_result.importances_mean,
    'std_dev': KNN_perm_result.importances_std
}).sort_values(by='importance', ascending=False)

RFC_perm_df

,feature,importance,std_dev
7,garden,0.104316,0.003714
0,zip_code,0.051144,0.003772
1,num_rooms,0.049568,0.002879
2,living_area_m2,0.047475,0.004062
5,terrace,0.019033,0.002147
8,garden_area_m2,0.000539,0.000649
11,num_bathrooms,0.000281,0.000911
9,num_facades,0.000086,0.000779
10,swimming_pool,-0.000022,0.000065
4,furnished,-0.000065,0.000274


### Extracting Coefficients (Logistic Regression)
Built in feature importance

In [18]:
# 1. Get feature names from the preprocessor
feature_names = logreg_model_pipeline.named_steps['preprocessor'].get_feature_names_out()

# 2. Get coefficients from the classifier
coefficients = logreg_model_pipeline.named_steps['classifier'].coef_[0]

# 3. Create a DataFrame
coef_df = pd.DataFrame({
    'feature': feature_names,
    'coef': coefficients
}).sort_values(by='coef', ascending=False).head(10)

print(coef_df)

             feature      coef
60  nom__zip_code_73  3.384820
48  nom__zip_code_56  3.357789
59  nom__zip_code_71  2.700612
53  nom__zip_code_65  2.694334
52  nom__zip_code_64  2.524513
47  nom__zip_code_55  2.515789
35  nom__zip_code_41  2.494607
39  nom__zip_code_45  2.436447
89       bin__garden  2.421271
57  nom__zip_code_69  2.185402


Checking land_surface_m2 and garden columns since they could be influencing the model decision too much


In [19]:
leakage_check = df.groupby('property_type').agg({
    'land_surface_m2': ['mean', 'min', 'max', 'count'],
    'garden': ['mean', 'sum'] 
})

print(leakage_check)

              land_surface_m2                          garden       
                         mean  min       max  count      mean    sum
property_type                                                       
Apartment            0.000000  0.0       0.0   9270  0.228263   2116
House             1192.676526  1.0  178367.0  12171  0.816390  11347


### CLASSIFICATION REPORT FOR ALL METRICS


In [20]:
from sklearn.metrics import classification_report

predictions = logreg_model_pipeline.predict(X_test)
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.88      0.85      0.86      1854
           1       0.90      0.92      0.91      2780

    accuracy                           0.89      4634
   macro avg       0.89      0.89      0.89      4634
weighted avg       0.89      0.89      0.89      4634



### 📈 Logistic Regression Model Progress Tracker

| Iteration | Key Modifications | Train Accuracy | Test Accuracy | Gap | Logic / Observation |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **1** | Baseline - basic preprocessing | 0.96 | 0.96 | 0 | Scores too high, model cheated because land_surface is 0 for all Apartments |
| **2** | Baseline - basic preprocessing - removed land_surface feature | 0.89 | 0.89 | 0 | without land_surface, models looks at other features to properly classify |
| **3** | Baseline - basic preprocessing - removed garden feature | 0.86 | 0.86 | 0 | Not much difference. Keeping garden feature |

> **Notes:** > * The **Gap** is the difference between Train and Test. A smaller gap means a more "trustworthy" model.
> * Best model so far:  **iteration 2**

# KNN


In [21]:
from sklearn.neighbors import KNeighborsClassifier

KNN_model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('KNN_classifier', KNeighborsClassifier())
])

Using RandomizedSearchCV to test out various K neighbors values

KNN needs so much compute, GridSearchCV nearly fried my PC

In [22]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'KNN_classifier__n_neighbors': [3, 5, 7, 9, 11],
    'KNN_classifier__weights': ['uniform'],
    'KNN_classifier__metric': ['euclidean', 'manhattan']
}

KNN_grid_search = RandomizedSearchCV(
    KNN_model_pipeline, 
    param_grid, 
    cv=5, 
    scoring='accuracy', 
    n_jobs=2,
    random_state=42)

KNN_grid_search.fit(X_train, y_train)

print(f"Best Score: {KNN_grid_search.best_score_}")
print(f"Best Params: {KNN_grid_search.best_params_}")

#he grid search automatically saves the winning model inside an attribute called best_estimator_
KNN_best_model = KNN_grid_search.best_estimator_
print(f"Best Model Train Score {KNN_best_model.score(X_train, y_train)}")
print(f"Best Model Test Score {KNN_best_model.score(X_test, y_test)}")

Best Score: 0.8861613164283787
Best Params: {'KNN_classifier__weights': 'uniform', 'KNN_classifier__n_neighbors': 9, 'KNN_classifier__metric': 'manhattan'}
Best Model Train Score 0.9062854059886701
Best Model Test Score 0.885843763487268


In [23]:
from sklearn.metrics import classification_report

predictions = KNN_best_model.predict(X_test)
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.82      0.91      0.86      1854
           1       0.93      0.87      0.90      2780

    accuracy                           0.89      4634
   macro avg       0.88      0.89      0.88      4634
weighted avg       0.89      0.89      0.89      4634



### Permutation Importance
To check which feature influenced the most

In [24]:
KNN_perm_result = permutation_importance(
    KNN_best_model, 
    X_test, 
    y_test, 
    scoring='accuracy', 
    n_repeats=10,
    random_state=42
)

RFC_perm_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance': KNN_perm_result.importances_mean,
    'std_dev': KNN_perm_result.importances_std
}).sort_values(by='importance', ascending=False).head()
RFC_perm_df

,feature,importance,std_dev
7,garden,0.095943,0.003130
0,zip_code,0.048295,0.004180
1,num_rooms,0.043505,0.003732
2,living_area_m2,0.026111,0.002847
9,num_facades,0.016012,0.003497


WHY?
1. Garden

Because KNN plots points in space. If a property has garden = 1, it gets physically thrown into a massive cluster of other properties with gardens. Since the vast majority of those are houses, the "neighborhood vote" almost always results in a House classification.

2. Zip code

If you drop a pin in the middle of Brussels (Zip 10), the 9 closest neighbors on the map are almost guaranteed to be apartments. If you drop a pin in rural Wallonia, the neighbors are houses.

3. num_rooms, living_area_m2, num_facades

If a property is in a mixed neighborhood (e.g., the suburbs), it looks at the facades and the area. A 4-facade property will be pushed closer to the "House" cluster, while a 2-facade property will be pulled toward the "Apartment" cluster.


### 📈 KNN Model Progress Tracker

| Iteration | Key Modifications | Train [score metric] | Test [score metric] | Gap | Logic / Observation |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **1** | Same preprocessing as LogReg. GridSearch: Best Params: {'KNN_classifier__weights': 'distance', 'KNN_classifier__n_neighbors': 9, 'KNN_classifier__metric': 'manhattan'}  | 0.99 | 0.88 | 11 | Extreme score in training. 'distance' param gave extreme voting power to physically closer neighbours |
| **2** | Same prepro as LogReg. GridSearch: removed 'distance' param option, other params same as 1st iteration | 0.90 | 0.88 | 0.2 | Smaller gap.  |
| **3** |  |  |  |  |  |

> **Notes:** > * The **Gap** is the difference between Train and Test. A smaller gap means a more "trustworthy" model.
> * Best model so far: **iteration 2**

# RandomForestClassifier 

In [25]:
from sklearn.ensemble import RandomForestClassifier

RFC_model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('RFC_classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_split=30,
        min_samples_leaf=10,
        class_weight='balanced'
    ))
])

In [26]:
RFC_model_pipeline.fit(X_train, y_train)
RFC_model_pipeline.score(X_train, y_train)

0.8945778257350958

In [27]:
RFC_model_pipeline.score(X_test, y_test)

0.8981441519205869

In [28]:
from sklearn.metrics import classification_report

predictions = RFC_model_pipeline.predict(X_test)
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.86      0.90      0.88      1854
           1       0.93      0.90      0.91      2780

    accuracy                           0.90      4634
   macro avg       0.89      0.90      0.89      4634
weighted avg       0.90      0.90      0.90      4634



In [29]:
from sklearn.inspection import permutation_importance

RFC_perm_result = permutation_importance(RFC_model_pipeline, 
                                         X_test, 
                                         y_test,
                                         scoring='accuracy',
                                         n_repeats=10,
                                         random_state=42 
)

RFC_perm_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance': RFC_perm_result.importances_mean,
    'std_dev': RFC_perm_result.importances_std
}).sort_values(by='importance', ascending=False).head()
RFC_perm_df

,feature,importance,std_dev
2,living_area_m2,0.063919,0.002859
7,garden,0.058092,0.002464
1,num_rooms,0.032348,0.002941
8,garden_area_m2,0.017846,0.002158
0,zip_code,0.013854,0.001246


### 📈 RandomForestClassifier Model Progress Tracker

| Iteration | Key Modifications | Train [score metric] | Test [score metric] | Gap | Logic / Observation |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **1** | Same prepro as other classifiers. Params: n_estimators=100, max_depth=15, min_samples_split=30 | 0.90 | 0.89 | 0.1 |  |
| **2** | Params: n_estimators=100, max_depth=15, min_samples_split=30, class_weight='balanced' | same | same |  | Thought class_weight would help since dataset is imbalanced |
| **3** | Added min_samples_leaf=5 then 10. Pushed max_depth to 20 | Same | Same |  | Hit Information Ceiling which is max logic the model can apply for the task |

> **Notes:** > * The **Gap** is the difference between Train and Test. A smaller gap means a more "trustworthy" model.
> * Best model so far: 